# Physics-Informed DeepONet — Inverse Problem

**Goal**: simultaneously reconstruct u(x, y, t) **and** infer the unknown tumor
growth rate `v` from sparse observations, using a DeepONet backbone with a
PDE-residual loss term.

## Key idea

Standard DeepONet (forward problem):
```
loss = MSE(û, u_data)
```

Physics-Informed DeepONet (inverse problem, this notebook):
```
loss = w_data * MSE(û, u_data)          # fit sparse observations
     + w_pde  * MSE(PDE_residual, 0)    # satisfy the PDE
     + w_ic   * MSE(û_t0, u_ic)         # match initial condition
     + w_bc   * MSE(û_boundary, 0)      # zero boundary condition
```

The parameter `v` is a learnable scalar embedded inside the PDE residual.
Backpropagation through the residual automatically tunes `v`.

## PDE (tumor growth, porous-medium / Fisher-KPP type)
```
u_t = 6 u (u_x² + u_y²) + 3 u² (u_xx + u_yy) + v u
```
where `v` is the unknown net proliferation rate to be inferred.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    np.random.seed(seed)

setup_seed(123456)

# ── device ──────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# 1. Load data
# ============================================================
data  = np.load('/content/drive/MyDrive/2D_RS_Real_1_t.npz')
t_arr = data['t']      # (T,)
x_arr = data['x']      # (Nx,)
y_arr = data['y']      # (Ny,)
usol  = data['u_sol']  # (Ny, Nx, T)

Nx, Ny, T = len(x_arr), len(y_arr), len(t_arr)

# Flat (x, y, t) table
xx, yy, tt = np.meshgrid(x_arr, y_arr, t_arr)
X_all = np.vstack((xx.ravel(), yy.ravel(), tt.ravel())).T  # (N_total, 3)
u_all = usol.flatten()[:, None]                            # (N_total, 1)
N_total = X_all.shape[0]

# Initial condition vector  (Branch Net sensor input)
u_ic      = usol[:, :, 0].flatten()   # (Ny*Nx,)
n_sensors = len(u_ic)

print(f'Grid        : Ny={Ny}, Nx={Nx}, T={T}')
print(f'Total points: {N_total}')
print(f'n_sensors   : {n_sensors}')
print(f'u range     : [{u_all.min():.4f}, {u_all.max():.4f}]')

def l2_relative_error(z_true, z_pred):
    return np.linalg.norm(z_true - z_pred) / np.linalg.norm(z_true)

In [ ]:
# ============================================================
# 2. Physics-Informed DeepONet architecture
# ============================================================
#
# Branch Net : encodes initial condition  u(x,y,0)  → b ∈ R^p
# Trunk Net  : encodes query point (x,y,t)           → τ ∈ R^p
# Output     : û = |b·τ + bias|
#
# The learnable scalar  v  lives outside the nets.
# It enters only via the PDE residual in the loss function,
# so the PDE drives its value — exactly like PINN.
# ============================================================

class BranchNet(torch.nn.Module):
    def __init__(self, n_in, hidden=128, p=64):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_in, hidden), torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden), torch.nn.Tanh(),
            torch.nn.Linear(hidden, p),
        )
    def forward(self, ic):
        return self.net(ic)   # (batch, p)


class TrunkNet(torch.nn.Module):
    def __init__(self, hidden=128, p=64):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(3, hidden), torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden), torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden), torch.nn.Tanh(),
            torch.nn.Linear(hidden, p), torch.nn.Tanh(),
        )
    def forward(self, xyt):
        return self.net(xyt)  # (batch, p)


class PIDeepONet(torch.nn.Module):
    """
    Physics-Informed DeepONet.

    Parameters
    ----------
    n_sensors : int   — length of initial-condition sensor vector
    hidden    : int   — width of hidden layers in both sub-nets
    p         : int   — output dimension (dot-product space)
    """
    def __init__(self, n_sensors, hidden=128, p=64):
        super().__init__()
        self.branch = BranchNet(n_sensors, hidden, p)
        self.trunk  = TrunkNet(hidden, p)
        self.bias   = torch.nn.Parameter(torch.zeros(1))

        # ── learnable physics parameter ──────────────────────
        # v is the unknown tumor proliferation rate.
        # Initialised near zero; the PDE loss will drive it to
        # its true value during training.
        self.v = torch.nn.Parameter(torch.tensor([0.0]))

    def forward(self, ic, xyt):
        """Return û(x,y,t).  ic and xyt must already require grad."""
        b   = self.branch(ic)    # (batch, p)
        tau = self.trunk(xyt)    # (batch, p)
        out = (b * tau).sum(dim=-1, keepdim=True) + self.bias
        return torch.abs(out)    # keep non-negative

print('PIDeepONet defined.')

In [ ]:
# ============================================================
# 3. Loss functions
# ============================================================

mse = torch.nn.MSELoss()

def grad(u, x):
    """First-order derivative of u w.r.t. x via autograd."""
    return torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]


def pde_residual_loss(model, ic_tensor, n_pde=2000):
    """
    Compute the PDE residual at n_pde random collocation points.

    PDE:  u_t = 6 u (u_x² + u_y²) + 3 u² (u_xx + u_yy) + v u

    Inputs must have requires_grad=True so autograd can compute
    the spatial and temporal derivatives.
    """
    # Random collocation points in [x∈-3,3] × [y∈-3,3] × [t∈0,1]
    x = (torch.rand(n_pde, 1, device=device) * 6 - 3).requires_grad_(True)
    y = (torch.rand(n_pde, 1, device=device) * 6 - 3).requires_grad_(True)
    t = (torch.rand(n_pde, 1, device=device)        ).requires_grad_(True)

    xyt = torch.cat([x, y, t], dim=1)           # (n_pde, 3)
    ic  = ic_tensor.expand(n_pde, -1)            # (n_pde, n_sensors)

    u     = model(ic, xyt)                       # û  (n_pde, 1)

    # Derivatives
    u_t   = grad(u, t)                           # ∂û/∂t
    u_x   = grad(u, x)                           # ∂û/∂x
    u_y   = grad(u, y)                           # ∂û/∂y
    u_xx  = grad(u_x, x)                         # ∂²û/∂x²
    u_yy  = grad(u_y, y)                         # ∂²û/∂y²

    v = model.v
    residual = (u_t
                - 6  * u  * (u_x**2 + u_y**2)
                - 3  * u**2 * (u_xx + u_yy)
                - v  * u)

    return mse(residual, torch.zeros_like(residual))


def ic_loss(model, ic_tensor, x_arr, y_arr, u_ic_np):
    """
    Enforce û(x, y, 0) ≈ u_ic at every sensor location.
    """
    xx0, yy0 = np.meshgrid(x_arr, y_arr)
    t0  = np.zeros_like(xx0.ravel())
    xyt0 = np.vstack([xx0.ravel(), yy0.ravel(), t0]).T  # (Ny*Nx, 3)

    xyt0_t  = torch.from_numpy(xyt0).to(device)
    u_ic_t  = torch.from_numpy(u_ic_np[:, None]).to(device)
    ic_exp  = ic_tensor.expand(xyt0_t.shape[0], -1)

    u_pred  = model(ic_exp, xyt0_t)
    return mse(u_pred, u_ic_t)


def bc_loss(model, ic_tensor, n_bc=200):
    """
    Zero Dirichlet BC on all four edges of [-3,3]².
    """
    t_vals  = torch.rand(n_bc, 1, device=device)
    edge    = torch.rand(n_bc, 1, device=device) * 6 - 3
    three   = 3 * torch.ones(n_bc, 1, device=device)
    m_three = -three

    # Four boundary segments
    pts = torch.cat([
        torch.cat([m_three, edge,   t_vals], 1),  # left
        torch.cat([three,   edge,   t_vals], 1),  # right
        torch.cat([edge,    m_three,t_vals], 1),  # bottom
        torch.cat([edge,    three,  t_vals], 1),  # top
    ], dim=0)                                      # (4*n_bc, 3)

    ic_exp = ic_tensor.expand(pts.shape[0], -1)
    u_pred = model(ic_exp, pts)
    return mse(u_pred, torch.zeros_like(u_pred))


def data_loss(model, ic_tensor, X_data, u_data):
    """
    Fit sparse interior observations.
    """
    ic_exp = ic_tensor.expand(X_data.shape[0], -1)
    u_pred = model(ic_exp, X_data)
    return mse(u_pred, u_data)

print('Loss functions defined.')

In [ ]:
# ============================================================
# 4. Prepare tensors
# ============================================================

# ── sensor tensor (Branch Net input) ────────────────────────
ic_tensor = torch.from_numpy(u_ic).unsqueeze(0).to(device)  # (1, n_sensors)

# ── sparse data points (simulate limited observations) ──────
# We use only N_data points for the data-fit term, mimicking a
# real scenario where full-field measurements are unavailable.
N_data = min(2000, N_total)
ids    = np.random.choice(N_total, N_data, replace=False)
X_data = torch.from_numpy(X_all[ids]).to(device)   # (N_data, 3)
u_data = torch.from_numpy(u_all[ids]).to(device)   # (N_data, 1)

# ── full dataset for evaluation ─────────────────────────────
X_test = torch.from_numpy(X_all).to(device)

print(f'Sensor vector length : {n_sensors}')
print(f'Sparse data points   : {N_data}')
print(f'Test points          : {N_total}')

In [ ]:
# ============================================================
# 5. Training  —  Physics-Informed DeepONet
# ============================================================
#
# Loss weights
# -----------
# w_data : fit sparse observations  (most important anchor)
# w_pde  : satisfy the PDE          (drives v toward truth)
# w_ic   : match initial condition
# w_bc   : zero boundary condition
#
# Strategy: start with heavier PDE weight so v gets pulled
# toward the physical value early, then balance.
# ============================================================

epochs  = 50000
lr      = 1e-3
w_data  = 5.0
w_pde   = 1.0
w_ic    = 2.0
w_bc    = 1.0
n_pde   = 1500   # collocation points per PDE evaluation

model = PIDeepONet(n_sensors=n_sensors, hidden=128, p=64).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=lr)

# Learning-rate schedule: halve every 10 000 epochs
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10000, gamma=0.5)

# ── history ─────────────────────────────────────────────────
loss_hist = []
l2_hist   = []
v_hist    = []
iter_hist = []

print(f'Initial v = {model.v.item():.4f}  (true value unknown until convergence)')
print('Training ...')
print(f'{"Epoch":>8}  {"Total loss":>12}  {"L2 error":>10}  {"v":>8}')
print('-' * 48)

for epoch in range(epochs):
    model.train()
    opt.zero_grad()

    # Individual loss terms
    l_data = data_loss(model, ic_tensor, X_data, u_data)
    l_pde  = pde_residual_loss(model, ic_tensor, n_pde)
    l_ic   = ic_loss(model, ic_tensor, x_arr, y_arr, u_ic)
    l_bc   = bc_loss(model, ic_tensor)

    loss = w_data * l_data + w_pde * l_pde + w_ic * l_ic + w_bc * l_bc
    loss.backward()

    # Gradient clipping prevents exploding gradients from high-order PDE terms
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    opt.step()
    scheduler.step()

    if (epoch + 1) % 1000 == 0 or epoch == 0:
        model.eval()
        with torch.no_grad():
            # Batch evaluation to avoid OOM
            BATCH  = 4096
            preds  = []
            for s in range(0, N_total, BATCH):
                e = min(s + BATCH, N_total)
                ic_b = ic_tensor.expand(e - s, -1)
                preds.append(model(ic_b, X_test[s:e]).cpu().numpy())
            u_pred_np = np.concatenate(preds, axis=0)

        err = l2_relative_error(u_all, u_pred_np)
        v_val = model.v.item()

        loss_hist.append(loss.item())
        l2_hist.append(err)
        v_hist.append(v_val)
        iter_hist.append(epoch + 1)

        print(f'{epoch+1:>8d}  {loss.item():>12.6f}  {err:>10.6f}  {v_val:>8.4f}')

print('\nTraining complete.')
print(f'Final v = {model.v.item():.6f}')

In [ ]:
# ============================================================
# 6. Results: v convergence + L2 error
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── v convergence ────────────────────────────────────────────
axes[0].plot(iter_hist, v_hist, 'b-o', markersize=3, linewidth=1.5, label='PI-DeepONet v')
axes[0].axhline(y=model.v.item(), color='b', linestyle='--', alpha=0.4, label=f'Final v = {model.v.item():.4f}')
# If you know the ground-truth v from the data-generation script,
# uncomment and set it here:
# TRUE_V = 1.0
# axes[0].axhline(y=TRUE_V, color='r', linestyle='--', label=f'True v = {TRUE_V}')
axes[0].set_xlabel('Iterations', fontsize=13)
axes[0].set_ylabel('Inferred v', fontsize=13)
axes[0].set_title('Tumor growth rate v — convergence', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True)

# ── L2 error ─────────────────────────────────────────────────
axes[1].plot(iter_hist, l2_hist, 'g-o', markersize=3, linewidth=1.5, label='PI-DeepONet')
axes[1].set_xlabel('Iterations', fontsize=13)
axes[1].set_ylabel('L2 relative error', fontsize=13)
axes[1].set_title('u(x,y,t) reconstruction error', fontsize=13)
axes[1].set_yscale('log')
axes[1].legend(fontsize=11)
axes[1].grid(True)

plt.suptitle('Physics-Informed DeepONet — Inverse Tumor Growth Problem', fontsize=14)
plt.tight_layout()
plt.show()

print(f'\nFinal inferred v : {model.v.item():.6f}')
print(f'Final L2 error   : {l2_hist[-1]:.6f}')

In [ ]:
# ============================================================
# 7. Three-way comparison plot
# (paste your PINN results here to enable comparison)
# ============================================================
#
# From your _2D_TG_v.ipynb run, copy:
#   pinn_iters   — list of iteration numbers where you recorded v
#   pinn_v_vals  — list of v values at those iterations
#   pinn_l2      — list of L2 errors
#
# Example (replace with your actual values):
# pinn_iters  = [1, 1000, 2000, ..., 80000]
# pinn_v_vals = [0.xx, 0.xx, ...]
# pinn_l2     = [0.xx, 0.xx, ...]

pinn_iters  = None
pinn_v_vals = None
pinn_l2     = None

if pinn_iters is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(iter_hist,  v_hist,     'b-o', markersize=3, label='PI-DeepONet')
    axes[0].plot(pinn_iters, pinn_v_vals,'r-o', markersize=3, label='PINN (SGLD)')
    axes[0].set_xlabel('Iterations'); axes[0].set_ylabel('Inferred v')
    axes[0].set_title('v convergence comparison'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(iter_hist,  l2_hist, 'b-o', markersize=3, label='PI-DeepONet')
    axes[1].plot(pinn_iters, pinn_l2, 'r-o', markersize=3, label='PINN (SGLD)')
    axes[1].set_xlabel('Iterations'); axes[1].set_ylabel('L2 relative error')
    axes[1].set_title('u reconstruction comparison')
    axes[1].set_yscale('log'); axes[1].legend(); axes[1].grid(True)

    plt.suptitle('PINN vs Physics-Informed DeepONet', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Paste PINN results into pinn_iters / pinn_v_vals / pinn_l2 to enable comparison.')

In [ ]:
# ============================================================
# 8. Spatial visualisation at a chosen time slice
# ============================================================
t_idx = -1   # change to 0, 1, 2 ... for earlier time steps

mask = np.isclose(X_all[:, 2], t_arr[t_idx])
X_sl = torch.from_numpy(X_all[mask]).to(device)
ic_sl = ic_tensor.expand(X_sl.shape[0], -1)

model.eval()
with torch.no_grad():
    u_pred_sl = model(ic_sl, X_sl).cpu().numpy().reshape(Ny, Nx)

u_true_sl = u_all[mask].reshape(Ny, Nx)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, field, title in zip(
    axes,
    [u_true_sl, u_pred_sl, np.abs(u_true_sl - u_pred_sl)],
    [f'Ground truth  t={t_arr[t_idx]:.2f}',
     f'PI-DeepONet   t={t_arr[t_idx]:.2f}  v={model.v.item():.4f}',
     'Absolute error']
):
    cmap = 'Blues' if 'error' in title.lower() else 'hot'
    im = ax.contourf(x_arr, y_arr, field, levels=50, cmap=cmap)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9. Loss breakdown — understand what each term contributes
# ============================================================
model.eval()
with torch.no_grad():
    ld = data_loss(model, ic_tensor, X_data, u_data).item()
    li = ic_loss(model, ic_tensor, x_arr, y_arr, u_ic).item()
    lb = bc_loss(model, ic_tensor).item()

# PDE loss needs grad so compute separately
lp = pde_residual_loss(model, ic_tensor, n_pde=3000).item()

print('Final loss breakdown')
print(f'  Data loss     : {ld:.6f}  (weight {w_data})')
print(f'  PDE loss      : {lp:.6f}  (weight {w_pde})')
print(f'  IC  loss      : {li:.6f}  (weight {w_ic})')
print(f'  BC  loss      : {lb:.6f}  (weight {w_bc})')
print(f'\nInferred v      : {model.v.item():.6f}')
print(f'Final L2 error  : {l2_hist[-1]:.6f}')